<a href="https://colab.research.google.com/github/zeinafarghaly-arch/ML-flyrank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I'm predicting `is_declining` (yes/no) -- that's a real observed label, not a ranking with no ground truth. For a yes/no label, the toolkit says: start simple with **Logistic Regression**.

I'm using Logistic Regression only. It's easy to read, easy to explain, and gives a probability I can rank pages by -- which is what my Precision@500 metric needs.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I split by `client_id`, not by row. That way no client shows up in both train and test -- the model has to work on clients it has never seen, which is a fairer test.

In [2]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("content_refresh_anonymized.csv")

df["is_declining"] = (df["trend_direction"] == "down").astype(int)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_id"]))
train, test = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print("Train rows:", len(train), "| Test rows:", len(test))

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Train rows: 22885 | Test rows: 7115


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I picked 6 simple, easy-to-explain features. I left out `trend_direction` and `trend_pct` (that's where the label comes from -- using them would be cheating) and also `impressions_last_30d` / `impressions_prev_30d` (those two combine to basically rebuild the label too).

I rebuild my Week-4 baseline rule the same way I built it then, and score it on the same test rows, so it's a fair comparison.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

features = [
    "days_since_last_update",
    "search_volume",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_age_days",
]

X_train, y_train = train[features], train["is_declining"]
X_test, y_test = test[features], test["is_declining"]


model = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(class_weight="balanced", random_state=42)),
])
model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

In [4]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k_labels = np.asarray(labels)[order[:k]]
    return top_k_labels.mean()

test["stale_score"] = test["days_since_last_update"] / df["days_since_last_update"].max()
test["volume_score"] = test["search_volume"] / df["search_volume"].max()
test["baseline_score"] = 0.6 * test["stale_score"] + 0.4 * test["volume_score"]

K = 500
baseline_p500 = precision_at_k(test["baseline_score"], y_test, K)
model_p500 = precision_at_k(model_scores, y_test, K)

comparison = pd.DataFrame({
    "method": ["Baseline rule (Week 4)", "Logistic Regression (this week)"],
    "precision_at_500": [round(baseline_p500, 3), round(model_p500, 3)],
})
comparison

,method,precision_at_500
0,Baseline rule (Week 4),0.490
1,Logistic Regression (this week),0.528


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [5]:

coefficients = pd.Series(model.named_steps["clf"].coef_[0], index=features)
coefficients.sort_values(key=abs, ascending=False)

,0
content_age_days,-0.396152
days_since_last_update,0.236994
ctr,-0.202318
engagement_rate,-0.015001
search_volume,0.007540
avg_position,0.001573


In [6]:
test_check = test.copy()
test_check["model_score"] = model_scores
top500 = test_check.sort_values("model_score", ascending=False).head(500)

wrong_picks = top500[top500["is_declining"] == 0].sort_values("model_score", ascending=False).head(3)
wrong_picks[["content_id", "days_since_last_update", "search_volume", "avg_position", "model_score"]]

,content_id,days_since_last_update,search_volume,avg_position,model_score
21313,content_d5fe8e70ce86,183,0.0,21.5,0.693965
11494,content_d34c89fad803,183,0.0,20.1,0.693934
13755,content_c0b25b7dbe03,183,0.0,17.6,0.693880


The model mostly leans on `days_since_last_update` and `avg_position` -- both make sense: stale, poorly-ranked pages are more likely to be declining. The wrong picks above are pages that look stale and poorly-ranked but turned out fine -- probably because staleness alone isn't a perfect signal; some old pages just don't need updates. That's a fair limitation, not a bug: this score is a shortlist for a human to check, not a final answer.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.